In [ ]:
# Queries specifically targeting patient/control designs

from Bio import Entrez
import pandas as pd

Entrez.email = "samireformed00786@gmail.com"



def fetch_summaries(id_list):
    if not id_list:          # guard against empty list
        return []
    ids = ",".join(id_list)
    handle = Entrez.esummary(db="gds", id=ids)
    records = Entrez.read(handle)
    handle.close()
    return records



neuro_patient_control = [
    # Parkinson's - direct patient vs control
    "Parkinson's disease AND control AND RNA-seq AND human",
    "Parkinson's disease AND healthy AND transcriptome AND blood",
    "Parkinson's disease AND substantia nigra AND control AND RNA-seq",
    "Parkinson's disease AND prefrontal cortex AND control",
    "GBA1 AND Parkinson AND control AND RNA-seq",
    "LRRK2 AND Parkinson AND control AND transcriptome",
    "SNCA AND Parkinson AND control AND RNA-seq",

    # Alzheimer's - well studied, lots of clean datasets  
    "Alzheimer's disease AND control AND RNA-seq AND human",
    "Alzheimer's disease AND hippocampus AND control AND RNA-seq",

    # ALS
    "amyotrophic lateral sclerosis AND control AND RNA-seq",

    # General neurodegeneration
    "neurodegeneration AND patient AND control AND RNA-seq",
]

def search_geo(query, retmax=100):
    try:
        handle = Entrez.esearch(db="gds", term=query, retmax=retmax, sort="pub+date")
        record = Entrez.read(handle)
        handle.close()
        return record['IdList']
    except Exception as e:
        print(f"    [search error: {e}]")
        return []
    
def to_dataframe(summaries):
    rows = []
    for s in summaries:
        n_samples = int(s.get('n_samples', 0))
        rows.append({
            'accession':  s.get('Accession', ''),
            'title':      s.get('title', ''),
            'organism':   s.get('taxon', ''),
            'n_samples':  n_samples,
            'date':       s.get('PDAT', ''),
            'has_pub':    len(s.get('PubMedIds', [])) > 0,
            'type':       s.get('gdsType', ''),
            'summary':    s.get('summary', '').lower(),
        })
    return pd.DataFrame(rows)

# ── run and filter ─────────────────────────────────────────────

all_results = []

for q in neuro_patient_control:
    ids = search_geo(q, retmax=100)
    print(f"  {q[:55]:<55} → {len(ids)} datasets")
    if ids:
        summaries = fetch_summaries(ids[:50])
        df = to_dataframe(summaries)
        df['query'] = q
        all_results.append(df)

master = pd.concat(all_results, ignore_index=True).drop_duplicates('accession')

# ── post-fetch filtering ───────────────────────────────────────

# keywords that indicate patient/control design in the summary
control_keywords = ['control', 'healthy', 'case', 'patient', 'disease vs', 'vs control', 'normal']

def has_control_design(summary_text):
    return any(kw in summary_text for kw in control_keywords)

filtered = master[
    (master['summary'].apply(has_control_design)) &  # mentions control/patient
    (master['n_samples'] >= 10) &                    # enough samples
    (master['n_samples'] <= 100) &                   # not overwhelmingly large
    (master['has_pub'] == False)                     # unpublished — unclaimed
].copy()

# Simple score: datasets with more samples are more reliable
filtered['score'] = filtered['n_samples'].apply(lambda x: min(x, 50))

filtered = filtered.sort_values('score', ascending=False)

cols = ['accession', 'title', 'organism', 'n_samples', 'date', 'query']
filtered[cols].to_csv('neuro_patient_control_unpublished.csv', index=False)

print(f"\nTotal unique datasets:          {len(master)}")
print(f"Patient/control unpublished:    {len(filtered)}")
print(f"Saved to neuro_patient_control_unpublished.csv")

  Parkinson's disease AND control AND RNA-seq AND human   → 78 datasets
